# TrainLM: start pretraining from raw text

Open this notebook in Colab or Kaggle with a **single-device native TorchTPU runtime**. By default TrainLM builds a randomly initialized GPT-2 model, learns a tokenizer from a pinned Hugging Face dataset, prepares packed data, trains, evaluates, and exports a normal Hugging Face model. The setup cell also lets you select another registered Transformers causal-LM architecture and pass its config fields.

For a first run, choose the token budget and sequence length. The default corpus is a pinned TinyStories revision. This is an experimental, small-scale workflow; it does not establish convergence or TPU performance.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-native-scratch-source-")) / "TrainLM"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch",
    "main",
    "https://github.com/Dhiraj309/TrainLM.git", str(repo_root),
], check=True)
from importlib.util import find_spec
missing = [
    requirement
    for module, requirement in (("datasets", "datasets>=3,<5"), ("tokenizers", "tokenizers>=0.21,<1"))
    if find_spec(module) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
print(f"source_revision={source_revision}")

## 1. Choose your run

Edit the token budget and sequence length below. You can also choose another Hugging Face dataset by setting its owner/name, immutable 40-character revision, separate train/validation splits, and text column. The setup cell installs only the raw-text `datasets` and `tokenizers` helpers if missing; it leaves the preinstalled Torch/TorchTPU stack alone. GPT-2 is the default. To use another built-in Transformers causal-LM architecture, set `MODEL_TYPE` and `MODEL_CONFIG_OVERRIDES` in the setup cell. TrainLM fills vocabulary and special-token IDs from its trained tokenizer and handles tokenization, data packing, runtime checks, and training arguments.

Runs use the stable `RUN_DIR` shown below, and TrainLM reuses verified tokenizer and packed-data artifacts when the pinned preparation settings match. To resume after an interruption, set `RESUME_FROM_CHECKPOINT = "latest"` and keep the same run settings. To deliberately extend a completed or interrupted run, increase `TARGET_SUPERVISED_TOKENS`, set `EXTEND_TRAINING_BUDGET = True`, and resume from `"latest"`. Keep the model, dataset, optimizer, precision, and runtime settings unchanged. Extension restores model and optimizer state, then starts a new cosine schedule at the configured learning rate over the remaining steps; this can raise the learning rate at the extension boundary. Do not set the extension flag for an ordinary exact resume. Choose a fresh `RUN_DIR` for changes to model, data, or other run settings. TrainLM saves at periodic intervals and writes a final checkpoint when a run completes successfully off the interval; an interruption can still require repeating work since the last completed periodic checkpoint. For Colab resets, mount Google Drive and point `RUN_DIR` under `/content/drive/MyDrive`; in Kaggle, save a notebook version to retain `/kaggle/working` outputs. CPU/API coverage does not validate native TPU checkpoint recovery.


TrainLM writes the effective run settings to `RUN_DIR/artifacts/run_config.json` and stores their fingerprint in every checkpoint. Exact resume requires matching model, data, optimizer, schedule, precision, and runtime settings. The explicit budget-extension mode permits only a larger token budget and its derived training horizon/cadence; other mismatches are rejected before checkpoint state is restored.

Keep up to three completed automatic checkpoints with `SAVE_TOTAL_LIMIT = 3`. Set it to `None` to keep all. Cleanup happens only after a successful save. Older checkpoints created before retention support, incomplete saves, and folders containing user files or exports are preserved. You can change this storage limit when resuming.


In [ ]:
from pathlib import Path
import json
import re
import subprocess
import sys

# Beginner settings
TARGET_SUPERVISED_TOKENS = 10_000
SEQUENCE_LENGTH = 128
# Storage limit for automatic checkpoints; None keeps all.
SAVE_TOTAL_LIMIT = 3

# For an existing run only: increase TARGET_SUPERVISED_TOKENS, then enable both.
EXTEND_TRAINING_BUDGET = False
# Set True to restore configuration and checkpoint from RUN_DIR/report.json.
RESUME_FROM_REPORT = False

# Optional data settings; keep the pinned defaults for your first run.
DATASET_ID = "roneneldan/TinyStories"
DATASET_REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
TRAIN_SPLIT = "train"
VALIDATION_SPLIT = "validation"
TEXT_FIELD = "text"

# Optional: keep GPT-2 for the starter run, or configure another built-in HF model.
MODEL_TYPE = "gpt2"
MODEL_CONFIG_OVERRIDES = None
# Example small Llama config (also set MODEL_TYPE = "llama" above):
# MODEL_CONFIG_OVERRIDES = {
#     "hidden_size": 512,
#     "intermediate_size": 1_376,
#     "num_hidden_layers": 8,
#     "num_attention_heads": 8,
#     "num_key_value_heads": 4,
#     "max_position_embeddings": SEQUENCE_LENGTH,
# }

# This path stays the same when you rerun cells in the current runtime.
RUN_DIR = Path("/content/trainlm-pretraining")
if Path("/kaggle/working").is_dir():
    RUN_DIR = Path("/kaggle/working/trainlm-pretraining")
# To keep Colab runs across runtime resets, mount Drive and set RUN_DIR to
# Path("/content/drive/MyDrive/TrainLM/pretraining"). Mount Drive before enabling
# RESUME_FROM_REPORT so the setup cell can read the saved report.
if RESUME_FROM_REPORT:
    report_path = RUN_DIR / "report.json"
    if not report_path.is_file():
        raise FileNotFoundError(
            f"Resume report is missing: {report_path}. Restore the original RUN_DIR."
        )
    saved_run_report = json.loads(report_path.read_text(encoding="utf-8"))
    if saved_run_report.get("schema_version") != 6:
        raise ValueError(
            "This recovery cell needs a schema-v6 report with resume metadata."
        )
    saved_source_revision = saved_run_report.get("source_revision")
    if not isinstance(saved_source_revision, str) or not re.fullmatch(
        r"[0-9a-f]{40}", saved_source_revision
    ):
        raise ValueError("The report has no valid TrainLM source revision.")
    subprocess.run(
        ["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", saved_source_revision],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(repo_root), "checkout", "--detach", "FETCH_HEAD"],
        check=True,
    )
    source_revision = subprocess.check_output(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
    ).strip()
    if source_revision != saved_source_revision:
        raise RuntimeError("TrainLM could not restore the source revision in report.json.")

sys.path.insert(0, str(repo_root / "src"))
from trainlm import TrainLMPretrainingConfig, TrainLMTrainer

RESUME_FROM_CHECKPOINT = None  # Set to "latest" to resume or extend.
# Extension also requires EXTEND_TRAINING_BUDGET = True above.

session_dir = RUN_DIR
session_dir.mkdir(parents=True, exist_ok=True)
output_dir = session_dir / "artifacts"
report_path = session_dir / "report.json"
pretraining_config = TrainLMPretrainingConfig(
    output_dir=output_dir,
    dataset_id=DATASET_ID,
    dataset_revision=DATASET_REVISION,
    train_split=TRAIN_SPLIT,
    validation_split=VALIDATION_SPLIT,
    text_field=TEXT_FIELD,
    model_type=MODEL_TYPE,
    model_config_overrides=MODEL_CONFIG_OVERRIDES,
    max_tokens=TARGET_SUPERVISED_TOKENS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    sequence_length=SEQUENCE_LENGTH,
    accelerator="native_tpu",
)
print(f"source_revision={source_revision}")
print(f"target_supervised_tokens={TARGET_SUPERVISED_TOKENS}")
print(f"output_dir={output_dir}")

## Optional: restore a run from its report

For a new Colab runtime, mount the same Drive folder first and keep `RUN_DIR` pointing to the original run. Set `RESUME_FROM_REPORT = True` in the settings cell, then run the notebook from the beginning. The setup pins the TrainLM source revision from `report.json`; this cell restores the saved config and checkpoint before the training cell runs. It also checks package versions and required run files. If any are missing or different, it stops with a message instead of starting a new run. The normal training cell then resumes, evaluates, exports, and refreshes the report. The report must include a published checkpoint.


In [ ]:
if RESUME_FROM_REPORT:
    from importlib.metadata import PackageNotFoundError, version

    resume_record = saved_run_report.get("resume") or {}
    if resume_record.get("status") != "available":
        raise RuntimeError(
            "This report has no published checkpoint to resume. "
            "Complete a training update first."
        )
    saved_packages = saved_run_report.get("runtime", {}).get("packages", {})
    package_mismatches = []
    for distribution, saved_version in saved_packages.items():
        if saved_version is None:
            continue
        try:
            installed_version = version(distribution)
        except PackageNotFoundError:
            installed_version = None
        if installed_version != saved_version:
            package_mismatches.append(
                f"{distribution}: report={saved_version}, installed={installed_version}"
            )
    if package_mismatches:
        raise RuntimeError(
            "Restore the runtime package versions recorded in report.json before "
            "resuming: " + "; ".join(package_mismatches)
        )

    saved_settings = resume_record.get("pretraining_config")
    if not isinstance(saved_settings, dict):
        raise ValueError("The report does not contain the saved pretraining config.")
    restored_config = TrainLMPretrainingConfig(**saved_settings)
    session_dir = Path(RUN_DIR).resolve()
    output_dir = Path(restored_config.output_dir).resolve()
    if output_dir.parent != session_dir:
        raise RuntimeError(
            "Set RUN_DIR to the original run folder recorded by the saved config."
        )
    checkpoint_value = resume_record.get("checkpoint_path")
    if not isinstance(checkpoint_value, str) or not checkpoint_value:
        raise ValueError("The report does not contain a checkpoint path.")
    checkpoint_input = Path(checkpoint_value).expanduser()
    if checkpoint_input.is_symlink():
        raise ValueError("The reported checkpoint cannot be a symlink.")
    resume_checkpoint = checkpoint_input.resolve()
    required_run_files = (
        output_dir / "run_config.json",
        output_dir / "preparation_cache.json",
    )
    if (
        not resume_checkpoint.is_relative_to(output_dir)
        or not resume_checkpoint.is_dir()
        or any(not item.is_file() for item in required_run_files)
    ):
        raise FileNotFoundError(
            "The original checkpoint, run config, or preparation cache is missing. "
            "Restore the complete RUN_DIR before resuming."
        )
    extension_flag = resume_record.get("extend_training_budget", False)
    if not isinstance(extension_flag, bool):
        raise ValueError("The report's budget-extension flag must be a boolean.")

    pretraining_config = restored_config
    output_dir = Path(restored_config.output_dir)
    report_path = session_dir / "report.json"
    TARGET_SUPERVISED_TOKENS = restored_config.max_tokens
    SEQUENCE_LENGTH = restored_config.sequence_length
    SAVE_TOTAL_LIMIT = restored_config.save_total_limit
    EXTEND_TRAINING_BUDGET = extension_flag
    DATASET_ID = restored_config.dataset_id
    DATASET_REVISION = restored_config.dataset_revision
    TRAIN_SPLIT = restored_config.train_split
    VALIDATION_SPLIT = restored_config.validation_split
    TEXT_FIELD = restored_config.text_field
    MODEL_TYPE = restored_config.model_type
    MODEL_CONFIG_OVERRIDES = dict(restored_config.model_config_overrides)
    RESUME_FROM_CHECKPOINT = resume_checkpoint
    print(f"resume_from_report={report_path}")
    print(f"resume_checkpoint={resume_checkpoint}")
    print(f"target_supervised_tokens={TARGET_SUPERVISED_TOKENS}")


## 2. Train and export

The public `TrainLMTrainer.from_scratch()` API prepares the tokenizer, packed data, and selected model architecture. Then use the familiar `train()`, `evaluate()`, and `save_model()` methods. Set `RESUME_FROM_CHECKPOINT = "latest"` for the newest completed checkpoint. For a larger token budget, increase `TARGET_SUPERVISED_TOKENS`, set `EXTEND_TRAINING_BUDGET = True`, and resume from `"latest"`; keep all other run settings unchanged. The API validates that only the token budget and derived horizon changed, restores the saved model and optimizer, then starts a new cosine schedule at the configured learning rate. This opt-in transition can increase the learning rate, unlike exact same-budget resume. After training, `trainer.latest_checkpoint` returns the newest completed checkpoint path or `None`; the output prints it for copying, and the downloaded report records it at `checkpoint.latest_path`. Set `RESUME_FROM_CHECKPOINT = "latest"` on the next run to resume it. Native TorchTPU remains experimental; CPU/API checks do not validate TPU checkpoint recovery.

When a checkpoint is available, the report also stores a ready-to-run `resume.code` example with every pretraining setting captured, including defaults. The download cell displays this Python snippet and includes `resume.py` in the ZIP. It resumes the recorded checkpoint with the same token budget; it does not add more training to a completed run. Restore the full original run directory at its recorded path and use the source revision and package versions recorded in the report.


In [ ]:
import hashlib
import json
import math
import platform
from importlib.metadata import PackageNotFoundError, version
from trainlm._notebook_resume import build_resume_report


def installed_version(distribution):
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None


runtime_report = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        "torch": installed_version("torch"),
        "transformers": installed_version("transformers"),
        "torch-tpu": installed_version("torch-tpu"),
        "libtpu": installed_version("libtpu"),
        "datasets": installed_version("datasets"),
        "tokenizers": installed_version("tokenizers"),
    },
}
model_report = {
    "requested_model_type": pretraining_config.model_type,
    "requested_config_overrides": dict(
        pretraining_config.model_config_overrides
    ),
}
tokenizer_report = {
    "algorithm": "byte_level_bpe",
    "training_split": TRAIN_SPLIT,
    "requested_vocab_size": pretraining_config.tokenizer_vocab_size,
}
data_report = {
    "dataset_id": pretraining_config.dataset_id,
    "dataset_revision": pretraining_config.dataset_revision,
    "train_split": pretraining_config.train_split,
    "validation_split": pretraining_config.validation_split,
    "text_field": pretraining_config.text_field,
    "max_train_examples": pretraining_config.max_train_examples,
    "max_eval_examples": pretraining_config.max_eval_examples,
}
training_config_report = {
    "requested": {
        "target_supervised_tokens": pretraining_config.max_tokens,
        "sequence_length": pretraining_config.sequence_length,
        "per_device_train_batch_size": pretraining_config.per_device_train_batch_size,
        "per_device_eval_batch_size": pretraining_config.per_device_eval_batch_size,
        "gradient_accumulation_steps": pretraining_config.gradient_accumulation_steps,
        "learning_rate": pretraining_config.learning_rate,
        "weight_decay": pretraining_config.weight_decay,
        "bf16": pretraining_config.bf16,
        "torch_compile": pretraining_config.torch_compile,
        "logging_steps": pretraining_config.logging_steps,
        "max_eval_batches": pretraining_config.max_eval_batches,
        "seed": pretraining_config.seed,
        "save_total_limit": pretraining_config.save_total_limit,
    },
}
accelerator_report = {"requested": pretraining_config.accelerator}
def checkpoint_report_for(trainer):
    if trainer is None:
        return {"latest_path": None}
    try:
        latest_checkpoint = trainer.latest_checkpoint
    except Exception as exc:
        return {"latest_path": None, "lookup_error_type": type(exc).__name__}
    return {"latest_path": str(latest_checkpoint) if latest_checkpoint else None}


def print_report_summary(report):
    training = report.get("training", {})
    effective = report.get("training_config", {}).get("effective", {})
    accelerator = report.get("accelerator", {})
    model = report.get("model", {})
    print(f"run_status={report.get('status', 'unknown')}")
    print(f"model={model.get('model_class') or model.get('requested_model_type', 'not_initialized')}")
    print(
        "training="
        f"steps_completed:{training.get('step', 'not_run')} "
        f"step_limit:{effective.get('max_steps', 'not_set')} "
        f"supervised_tokens:{training.get('supervised_tokens', 'not_run')} "
        f"target_tokens:{training.get('target_supervised_tokens', effective.get('max_tokens', 'not_set'))}"
    )
    device = accelerator.get("device_name") or accelerator.get("device_type") or "not_resolved"
    print(
        "accelerator="
        f"{accelerator.get('backend', accelerator.get('requested', 'not_resolved'))} "
        f"device:{device} world_size:{accelerator.get('world_size', 'not_resolved')} "
        f"runtime:{accelerator.get('runtime_status', 'not_resolved')} "
        f"properties:{accelerator.get('properties_query_status', 'not_queried')}"
    )
    checkpoint = report.get("checkpoint", {})
    print(f"latest_checkpoint={checkpoint.get('latest_path') or 'none'}")
    if checkpoint.get("lookup_error_type"):
        print(f"latest_checkpoint_lookup_error={checkpoint['lookup_error_type']}")
    resume = report.get("resume", {})
    print(f"resume_script={resume.get('script') or resume.get('reason', 'unavailable')}")


checkpoint_report = {"latest_path": None}
trainer = None
try:
    trainer = TrainLMTrainer.from_scratch(
        pretraining_config,
        extend_training_budget=EXTEND_TRAINING_BUDGET,
    )
    model = trainer.model
    tokenizer = trainer.processing_class
    parameters = list(model.parameters())
    model_report.update({
        "model_type": model.config.model_type,
        "model_class": type(model).__name__,
        "vocab_size": int(model.config.vocab_size),
        "tokenizer_vocab_size": len(tokenizer),
        "total_parameters": sum(
            int(parameter.numel()) for parameter in parameters
        ),
        "trainable_parameters": sum(
            int(parameter.numel())
            for parameter in parameters
            if parameter.requires_grad
        ),
    })
    tokenizer_report.update({
        "class": type(tokenizer).__name__,
        "base_vocab_size": int(tokenizer.vocab_size),
        "vocab_size": len(tokenizer),
        "special_tokens": {
            name: {
                "token": (
                    str(getattr(tokenizer, f"{name}_token"))
                    if getattr(tokenizer, f"{name}_token", None) is not None
                    else None
                ),
                "id": (
                    int(getattr(tokenizer, f"{name}_token_id"))
                    if getattr(tokenizer, f"{name}_token_id", None) is not None
                    else None
                ),
            }
            for name in ("unk", "pad", "bos", "eos")
        },
    })
    args = trainer.args
    optimizer = trainer.optimizer
    training_config_report["effective"] = {
        "accelerator": args.accelerator,
        "max_steps": args.max_steps,
        "max_tokens": args.max_tokens,
        "per_device_train_batch_size": args.per_device_train_batch_size,
        "per_device_eval_batch_size": args.per_device_eval_batch_size,
        "sequence_length": args.sequence_length,
        "gradient_accumulation_steps": args.gradient_accumulation_steps,
        "learning_rate": args.learning_rate,
        "betas": [float(value) for value in args.betas],
        "eps": args.eps,
        "weight_decay": args.weight_decay,
        "lr_scheduler_type": args.lr_scheduler_type,
        "warmup_steps": args.warmup_steps,
        "bf16": args.bf16,
        "fp16": args.fp16,
        "torch_compile": args.torch_compile,
        "logging_steps": args.logging_steps,
        "save_steps": args.save_steps,
        "save_total_limit": args.save_total_limit,
        "eval_steps": args.eval_steps,
        "max_eval_batches": args.max_eval_batches,
        "dataloader_num_workers": args.dataloader_num_workers,
        "dataloader_pin_memory": args.dataloader_pin_memory,
        "seed": args.seed,
        "report_to": args.report_to if isinstance(args.report_to, str) else list(args.report_to),
        "logging_verbosity": args.logging_verbosity,
        "loss_implementation": args.loss_implementation,
        "logits_chunk_size": args.logits_chunk_size,
        "max_grad_norm": trainer.engine.config.trainer.max_grad_norm,
        "optimizer": {
            "class": type(optimizer).__name__,
            "parameter_groups": [
                {
                    "learning_rate": float(group["lr"]),
                    "betas": [float(value) for value in group["betas"]],
                    "eps": float(group["eps"]),
                    "weight_decay": float(group["weight_decay"]),
                    "fused": bool(group.get("fused", False)),
                    "parameter_count": sum(
                        int(parameter.numel()) for parameter in group["params"]
                    ),
                }
                for group in optimizer.param_groups
            ],
        },
        "scheduler_class": type(trainer.scheduler).__name__,
    }
    diagnostics = trainer.runtime.diagnostics()
    try:
        import torch
        from trainlm.runtime.torchtpu_discovery import (
            discover_tpu,
            normalize_worker_report,
        )

        hardware = normalize_worker_report(
            discover_tpu(
                torch, verify_allocation=False, include_properties=True
            )
        )
    except Exception as exc:
        hardware = {"status": "query_failed", "error_type": type(exc).__name__}
    runtime_values = diagnostics.values
    accelerator_report.update({
        "backend": diagnostics.backend,
        "device_type": diagnostics.device_type,
        "precision": diagnostics.precision,
        "world_size": diagnostics.world_size,
        "rank": diagnostics.rank,
        "local_rank": diagnostics.local_rank,
        "runtime_status": runtime_values.get("native_tpu_status"),
        "device_count_visible": runtime_values.get("device_count_visible"),
        "torch_compile_enabled": runtime_values.get("compile_enabled"),
        "torch_tpu_version": runtime_values.get("torch_tpu_version"),
        "libtpu_version": runtime_values.get("libtpu_version"),
        "discovery_error_type": hardware.get("error_type"),
        "properties_source": hardware.get("source"),
        "properties_query_status": hardware.get("status"),
        "properties_allocation": hardware.get("allocation"),
        "device_name": hardware.get("device_name"),
        "topology": hardware.get("topology"),
        "hbm_bytes_per_device": hardware.get("hbm_bytes_per_device"),
        "hbm_bytes_per_chip": hardware.get("hbm_bytes_per_chip"),
        "devices_per_chip": hardware.get("devices_per_chip"),
        "properties_error_type": hardware.get("properties_error_type"),
    })
    print(f"resume_from_checkpoint={RESUME_FROM_CHECKPOINT or 'none'}")
    state = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
    checkpoint_report = checkpoint_report_for(trainer)
    evaluation = trainer.evaluate()
    export_dir = trainer.save_model(output_dir / "baseline-export")
    # Read the exported files so the report matches the artifact in the ZIP.
    export_path = Path(export_dir)
    model_report["effective_config"] = json.loads(
        (export_path / "config.json").read_text()
    )
    tokenizer_report["effective_config"] = json.loads(
        (export_path / "tokenizer_config.json").read_text()
    )
    tokenizer_report["tokenizer_json_sha256"] = hashlib.sha256(
        (export_path / "tokenizer.json").read_bytes()
    ).hexdigest()
    train_loss = float(state.loss) if state.loss is not None else None
    eval_loss = float(evaluation["eval_loss"])
    status = (
        "passed"
        if state.tokens_seen >= TARGET_SUPERVISED_TOKENS
        and train_loss is not None and math.isfinite(train_loss)
        and math.isfinite(eval_loss)
        else "failed"
    )
    report = {
        "schema_version": 6,
        "scope": "experimental_single_device_raw_text_pretraining",
        "status": status,
        "source_revision": source_revision,
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "device": "native_tpu",
        "runtime": runtime_report,
        "data": data_report,
        "model": model_report,
        "tokenizer": tokenizer_report,
        "training_config": training_config_report,
        "accelerator": accelerator_report,
        "checkpoint": checkpoint_report,
        "resume": build_resume_report(
            pretraining_config, trainer=trainer, checkpoint=checkpoint_report,
        ),
        "training": {
            "extend_training_budget": EXTEND_TRAINING_BUDGET,
            "resume_from_checkpoint": (
                str(RESUME_FROM_CHECKPOINT)
                if RESUME_FROM_CHECKPOINT is not None else None
            ),
            "step": state.step,
            "supervised_tokens": state.tokens_seen,
            "target_supervised_tokens": TARGET_SUPERVISED_TOKENS,
            "train_loss": train_loss,
            "eval_loss": eval_loss,
        },
        "model_export": str(export_dir),
        "limitations": ["experimental_single_device", "does_not_certify_performance"],
    }
except Exception as exc:
    if trainer is not None:
        checkpoint_report = checkpoint_report_for(trainer)
    report = {
        "schema_version": 6,
        "scope": "experimental_single_device_raw_text_pretraining",
        "status": "failed",
        "source_revision": source_revision,
        "runtime": runtime_report,
        "data": data_report,
        "model": model_report,
        "tokenizer": tokenizer_report,
        "training_config": training_config_report,
        "accelerator": accelerator_report,
        "checkpoint": checkpoint_report,
        "resume": build_resume_report(
            pretraining_config, trainer=trainer, checkpoint=checkpoint_report,
        ),
        "error_type": type(exc).__name__,
        "limitations": ["experimental_single_device", "does_not_certify_performance"],
    }
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
    print_report_summary(report)
    print(json.dumps(report, indent=2))
    raise RuntimeError(f"Pretraining failed; download the report at {report_path}.") from None

report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n")
print_report_summary(report)
print(json.dumps(report, indent=2))
print(f"shareable_report={report_path}")
if report["status"] != "passed":
    raise RuntimeError(f"Training did not meet its token/loss checks; see {report_path}.")

## 3. Download the result

The ZIP contains the Hugging Face model/tokenizer export and run report. The report records requested and effective training settings, optimizer and scheduler details, dataset limits, resolved backend, runtime TPU discovery and bounded device properties, model/tokenizer provenance, and training metrics. The cell prints a short run summary before the full JSON. Property discovery is non-allocating and labeled unverified. Colab starts a download; Kaggle saves the ZIP under `/kaggle/working` for the notebook output panel. If preflight or training fails, run the next cell to download the partial report without a model export.

If a checkpoint was published, the ZIP also contains `resume.py` and the cell displays its code for copying. The script uses the saved pretraining configuration and checkpoint path. Keep the original run directory (tokenizer, packed data, run metadata, and optimizer checkpoints); the ZIP contains the model export and report, not this full training state. With no checkpoint or a failed checkpoint lookup, `resume.code` is `null` and no resume script is included.


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

export_dir = session_dir / "artifacts" / "baseline-export"
required = ("config.json", "tokenizer_config.json")
missing = [name for name in required if not (export_dir / name).is_file()]
weights_present = (
    any(path.is_file() and path.suffix in {".safetensors", ".bin"}
        for path in export_dir.rglob("*"))
    if export_dir.is_dir() else False
)
model_export_ready = not missing and weights_present
if not report_path.is_file():
    raise RuntimeError("The evidence report is missing.")

resume_code = json.loads(report_path.read_text()).get("resume", {}).get("code")
artifact_bundle = session_dir / "trainlm-scratch-artifact.zip"
with zipfile.ZipFile(artifact_bundle, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(report_path, arcname="report.json")
    if resume_code:
        bundle.writestr("resume.py", resume_code)
    if model_export_ready:
        for path in sorted(export_dir.rglob("*")):
            if path.is_file():
                bundle.write(path, arcname=Path("model") / path.relative_to(export_dir))
if model_export_ready:
    print("model_export=included")
else:
    print(f"model_export=not_included missing_files={missing} weights_present={weights_present}")

if resume_code:
    from IPython.display import Code, display
    display(Code(resume_code, language="python"))

kaggle_output = Path("/kaggle/working")
if kaggle_output.is_dir():
    kaggle_artifact = kaggle_output / artifact_bundle.name
    shutil.copy2(artifact_bundle, kaggle_artifact)
    print(f"kaggle_artifact={kaggle_artifact}")
try:
    from google.colab import files
except ImportError:
    print(f"artifact_bundle={artifact_bundle}")
else:
    files.download(str(artifact_bundle))